# **Customizing Short-term Memory**

## **What's Covered?**
1. Introduction to Customizing Agent State
    - What is AgentState?
    - Default Structure of AgentState
    - The "messages" Key and What it Contains
2. Extending AgentState (Custom Schema)
    - Step 1: Define a Custom State
    - Step 2: Pass the State to `create_agent()`
    - Step 3: Call the Agent
    - Why Custom State Schema Fields Are Not Accessible to the LLM
3. Accessing Custom State from Tool via `ToolRuntime`
    - Step 1: Define a Custom State
    - Step 2: Define a Tool to access the state via `ToolRuntime`
    - Step 3: Pass the State in `create_agent()` using `state_schema` Arg
    - Step 4: Invoke the Agent
4. Modifying Agent State from Tool via `ToolRuntime`
    - Combining State Update + Control Flow with `Command`
    - What `Command` Enables
    - Step 1: Define a Custom State
    - Step 2: Define a Tool to access and `modify` the state via `ToolRuntime`
    - Step 3: Pass the State in `create_agent()` using `state_schema` Arg
    - Step 4: Invoke the Agent

## **Introduction to Customizing Agent State**

- Agents maintain conversation history automatically through the message state. We can configure the agent to use a custom state schema to remember additional information during the conversation.
- By default, our agent state tracks the list of messages only. But we can add custom fields.
- Note that the agent use **AgentState** to manage short term memory, specifically the conversation history via a **messages** key. 

Custom state schemas must extend `AgentState` as a TypedDict and passed to `state_schema` in create_agent().

### **What is AgentState?**
AgentState is the structured representation of the agent’s state during execution.

It acts as:
- Short-term memory (thread-level)
- A shared data layer across all components:
    - LLM (model)
    - tools
    - middleware
    - graph nodes

### **Default Structure of AgentState**

By default, AgentState contains:
```
{
    "messages": List[BaseMessage]
}
```

### **The "messages" Key and What it Contains**
The messages field is the core component of AgentState.

It contains
- `HumanMessage` - user input
- `AIMessage` - model responses
- `ToolMessage` - tool outputs

There are two ways to define custom state:
- Via middleware (preferred - Discussed in the next notebook)
- Via state_schema on create_agent

Defining custom state via middleware is preferred over defining it via state_schema on create_agent because it allows you to keep state extensions conceptually scoped to the relevant middleware and tools.

**[Click here for the details](https://docs.langchain.com/oss/python/langchain/agents#memory)**

## **Extending AgentState (Custom Schema)**

### **Step 1: Define a Custom State**

Extend Custom State with `AgentState` to add additional fields.

We can also extend the custom state with `TypeDict`. As of langchain v1.x, **custom state schema must be TypedDict types**. Pydantic models and dataclasses are no longer supported for custom `state_schema`.

Defining custom state via middleware is preferred over defining it via state_schema on create_agent because it allows you to keep state extensions conceptually scoped to the relevant middleware and tools.

In [1]:
# Step 1: Define a custom state
from langchain.agents import AgentState

class CustomAgentState(AgentState):  
    user_id: str
    theme_preference: str

### **Step 2: Pass the State to create_agent()**

Custom state schemas are passed to **create_agent()** using the **state_schema** parameter

In [3]:
from langchain_openai import ChatOpenAI

# Setup API Key
f = open('keys/.openai_api_key.txt')
OPENAI_API_KEY = f.read()

openai_chat_model = ChatOpenAI(
    api_key=OPENAI_API_KEY, 
    model="gpt-4o-mini", 
    temperature=1
)

In [4]:
# Step 2: Pass the State in create_agent using state_schema arg
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model=openai_chat_model,
    checkpointer=InMemorySaver(),
    state_schema=CustomAgentState
)

### **Step 3: Call the Agent**

In [5]:
# Step 3: Invoke Agent
from langchain.messages import HumanMessage

response = agent.invoke(
    {
        "messages": [HumanMessage(content="Hi! My name is Bob. Can you provide my preferences?")],
        "user_id": "user_123",
        "theme_preference": "dark"
    }, 
    {
        "configurable" : {"thread_id" : "1"}
    }
)

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

Hi! My name is Bob. Can you provide my preferences?
================================== Ai Message ==================================

Hi Bob! I can't access personal data about you unless you share it with me. However, if you tell me what your preferences are or what kind of information you're looking for, I'll do my best to help you!


### **Why Custom State Schema Fields Are Not Accessible to the LLM**

LLM only sees `state["messages"]`. LLM cannot see the custom state.

This is because LangChain want to avoid sending unnecessary data to LLM and reduce token usage. 

Note that, we can access this custom state inside:
- a tool
- a middleware

In [6]:
response

{'messages': [HumanMessage(content='Hi! My name is Bob. Can you provide my preferences?', additional_kwargs={}, response_metadata={}, id='3d291101-5888-44ea-bbd9-f58d06d1103c'),
  AIMessage(content="Hi Bob! I can't access personal data about you unless you share it with me. However, if you tell me what your preferences are or what kind of information you're looking for, I'll do my best to help you!", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 44, 'prompt_tokens': 20, 'total_tokens': 64, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_a64aa7d0ff', 'id': 'chatcmpl-DY8vmREykXqxUZhjAV0dDpB6CACjJ', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019dbf3b-b051-

## **Accessing Custom State from Tool via `ToolRuntime`**

Note that, in an agent, LLMs context will be:
- system_instruction and
- list of messages stored inside the state.

In order for LLM to recieve custom state variables, a state-aware tool is required. Let's see how to access agent state from a tool.

Access short term memory (state) in a tool using the **runtime** via **TooRuntime**. 

**Important**
> The runtime parameter is hidden from the tool signature (so the model doesn’t see it), but the tool can access the state through it.

### **Step 1: Define a Custom State**

In [1]:
# Step 1: Define a custom state
from langchain.agents import AgentState

class CustomAgentState(AgentState):  
    user_id: str
    theme_preference: str

### **Step 2: Define a Tool to `access` the state via `ToolRuntime`**

Remember that, using `ToolRuntime`, tool gets a direct access to the following:
- state
- store
- context
- etc...

In [10]:
# Step 2: Define a Tool to access the state via ToolRuntime
from langchain.tools import tool, ToolRuntime

@tool
def get_user_info(
    runtime: ToolRuntime
) -> str:
    """Look up user preferences."""
    user_id = runtime.state.get('user_id')
    theme_preference = runtime.state.get('theme_preference')
    return f"User preferences for {user_id} are: {theme_preference}"

### **Step 3: Pass the State in `create_agent()` using `state_schema` Arg**

In [7]:
from langchain_openai import ChatOpenAI

# Setup API Key
f = open('keys/.openai_api_key.txt')
OPENAI_API_KEY = f.read()

openai_chat_model = ChatOpenAI(
    api_key=OPENAI_API_KEY, 
    model="gpt-4o-mini", 
    temperature=1
)

In [8]:
# Step 3: Pass the State in create_agent using state_schema arg
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model=openai_chat_model,
    tools=[get_user_info],
    checkpointer=InMemorySaver(),
    state_schema=CustomAgentState
)

### **Step 4: Invoke the Agent**

In [9]:
# Step 4: Invoke Agent
from langchain.messages import HumanMessage

response = agent.invoke(
    {
        "messages": [HumanMessage(content="Hi! My name is Bob. Can you provide my preferences?")],
        "user_id": "user_123",
        "theme_preference": "dark"
    }, 
    {
        "configurable" : {"thread_id" : "1"}
    }
)

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

Hi! My name is Bob. Can you provide my preferences?
================================== Ai Message ==================================
Tool Calls:
  get_user_info (call_I5ddVh37k67m9NkOLQw6SvVP)
 Call ID: call_I5ddVh37k67m9NkOLQw6SvVP
  Args:
================================= Tool Message =================================
Name: get_user_info

User preferences for user_123 are: dark
================================== Ai Message ==================================

Hi Bob! Your preference is set to dark mode. If you have any other preferences or questions, feel free to ask!


## **Modifying Agent State from Tool via `ToolRuntime`**

In many real-world scenarios, tools also need to update the agent state. For example:
- storing results from external APIs
- updating intermediate workflow data
- enriching structured state for downstream steps

This is where `ToolRuntime` and `Command` come into play.

### **Combining State Update + Control Flow with `Command`**

LangGraph provides `Command` to combine state updates and execution routing in a single return value.

Normally langgraph nodes just **return state updates**, but `Command` **combines** that **with a goto** to control execution flow without needing extra edges. Nodes return `Command(update={...}, goto="next_node")` to update state AND route directly.

```python
from langgraph.types import Command

def my_node(state: dict) -> Command[Literal["my_other_node"]]:
    # Update state AND jump to another node
    return Command(
        update={"foo": "bar"},      # State changes
        goto="my_other_node"        # Control flow
    )
```
Use typed literals for goto like `Literal["node1", "node2"]` to enforce valid targets at runtime.

`langgraph.types.Command` lets LangGraph nodes return both state updates and control flow decisions (like jumping to another node) from a single function.

### **What `Command` Enables**
- **State Update** - Modify AgentState via `update={...}`
- **Control Flow** - Redirect execution using `goto="node_name"`
- **Type Safety with Literal** - Only valid node names can be used. Errors are caught early (at development)

### **Step 1: Define a Custom State**

In [2]:
# Step 1: Define a custom state
from langchain.agents import AgentState

class CustomAgentState(AgentState):  
    user_id: str
    theme_preference: str

### **Step 2: Define a Tool to access and `modify` the state via `ToolRuntime`**

In [3]:
from langchain.tools import tool, ToolRuntime

@tool
def get_user_info(
    runtime: ToolRuntime
) -> str:
    """Look up user preferences."""
    user_id = runtime.state.get('user_id')
    theme_preference = runtime.state.get('theme_preference')
    return f"User preferences for {user_id} are: {theme_preference}"

In [4]:
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def update_user_pref(
    updated_theme_preference: str,
    runtime: ToolRuntime
) -> Command:
    """
    Update the theme preference of the user in the state once they've revealed it. 
    """    
    return Command(
        update={
            "theme_preference": updated_theme_preference,
            "messages": [
                ToolMessage(f"Updated preferences successfully.", tool_call_id=runtime.tool_call_id)
            ]
        }
    )

### **Step 3: Pass the State in `create_agent()` using `state_schema` Arg**

In [5]:
from langchain_openai import ChatOpenAI

# Setup API Key
f = open('keys/.openai_api_key.txt')
OPENAI_API_KEY = f.read()

openai_chat_model = ChatOpenAI(
    api_key=OPENAI_API_KEY, 
    model="gpt-4o-mini", 
    temperature=1
)

In [6]:
# Step 3: Pass the State in create_agent using state_schema arg
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model=openai_chat_model,
    tools=[get_user_info, update_user_pref],
    checkpointer=InMemorySaver(),
    state_schema=CustomAgentState
)

### **Step 4: Invoke the Agent**

In [7]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {
        "messages": [HumanMessage(content="Hi! My name is Bob")],
        "user_id": "user_123",
        "theme_preference": "dark"
    }, 
    {
        "configurable" : {"thread_id" : "1"}
    }
)

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

Hi! My name is Bob
================================== Ai Message ==================================

Hello, Bob! How can I assist you today?


In [8]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {
        "messages": [HumanMessage(content="Can you provide my information")],
    }, 
    {
        "configurable" : {"thread_id" : "1"}
    }
)

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

Hi! My name is Bob
================================== Ai Message ==================================

Hello, Bob! How can I assist you today?
================================ Human Message =================================

Can you provide my information
================================== Ai Message ==================================
Tool Calls:
  get_user_info (call_oqk96PtkaxoLNPajqMFdf0KL)
 Call ID: call_oqk96PtkaxoLNPajqMFdf0KL
  Args:
================================= Tool Message =================================
Name: get_user_info

User preferences for user_123 are: dark
================================== Ai Message ==================================

Your current preference is set to "dark" theme. Is there anything else you'd like to know or update?


In [9]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {
        "messages": [HumanMessage(content="I dont like dark theme any more. I like a light one.")],
    }, 
    {
        "configurable" : {"thread_id" : "1"}
    }
)

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

Hi! My name is Bob
================================== Ai Message ==================================

Hello, Bob! How can I assist you today?
================================ Human Message =================================

Can you provide my information
================================== Ai Message ==================================
Tool Calls:
  get_user_info (call_oqk96PtkaxoLNPajqMFdf0KL)
 Call ID: call_oqk96PtkaxoLNPajqMFdf0KL
  Args:
================================= Tool Message =================================
Name: get_user_info

User preferences for user_123 are: dark
================================== Ai Message ==================================

Your current preference is set to "dark" theme. Is there anything else you'd like to know or update?
================================ Human Message =================================

I dont like dark theme any more. I like a light one.
===============

In [10]:
print(response["user_id"])
print(response["theme_preference"])

user_123
light


In [11]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {
        "messages": [HumanMessage(content="Can you provide my information? Also do you remember my name?")],
    }, 
    {
        "configurable" : {"thread_id" : "1"}
    }
)

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

Hi! My name is Bob
================================== Ai Message ==================================

Hello, Bob! How can I assist you today?
================================ Human Message =================================

Can you provide my information
================================== Ai Message ==================================
Tool Calls:
  get_user_info (call_oqk96PtkaxoLNPajqMFdf0KL)
 Call ID: call_oqk96PtkaxoLNPajqMFdf0KL
  Args:
================================= Tool Message =================================
Name: get_user_info

User preferences for user_123 are: dark
================================== Ai Message ==================================

Your current preference is set to "dark" theme. Is there anything else you'd like to know or update?
================================ Human Message =================================

I dont like dark theme any more. I like a light one.
===============